# Notebook 1: Method Overview

The goal of this notebook is to make the project readable and reproducible before running heavier experiments. The pipeline follows the same white-box intuition as RepE: if a model internally represents truth-related information, we should be able to extract that signal from hidden states with linear probes.

## The main question

Can we recover a linearly accessible "truthfulness" signal from hidden states, using the same prompt framing as RepE?

In this project, I combine lightweight grouped datasets with one dataset pulled from the RepE repository (`truthful.jsonl`).

- `cities`: factual geography questions
- `larger_than`: simple comparison questions
- `qa`: short factual questions
- `repeng_truthful`: honest vs dishonest self-report statements from RepE

Each group contains candidate answers where exactly one candidate is marked as correct. The probe is evaluated at the **group level**: it must rank the true candidate above the false candidate(s).

In [ ]:
from pathlib import Path
import sys

# Make the local package importable when the notebook is opened from notebooks/.
project_root = Path.cwd().parent
src_path = project_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from lie_detector_llm.datasets import build_dataset_collection

# Build the benchmark collection, including the RepE truthful dataset if present.
collection = build_dataset_collection(include_repeng_truthful=True)

# First look at the tabular structure used by the probe pipeline.
collection.frame.head()

,dataset_name,group_id,question,answer,prompt,label
0,cities,cities::Nairobi,Which country contains the city Nairobi?,Kenya,Consider the correctness of the answer to the ...,True
1,cities,cities::Tokyo,Which country contains the city Tokyo?,Thailand,Consider the correctness of the answer to the ...,False
2,cities,cities::Bangkok,Which country contains the city Bangkok?,Vietnam,Consider the correctness of the answer to the ...,False
3,larger_than,larger_than::18::42,Is 18 larger than 42?,No,Consider the correctness of the answer to the ...,True
4,cities,cities::Ottawa,Which country contains the city Ottawa?,Ireland,Consider the correctness of the answer to the ...,False


## Why the prompts are written this way

The prompt format is borrowed from the RepE setup. Instead of asking the model to directly answer the question, we ask it to judge whether a proposed answer is correct.

```text
Consider the correctness of the answer to the following question:

Question: ...
Answer: ...
The probability of the answer being correct is
```

That framing is useful because it pushes the model toward an internal correctness judgment. For this project, that is exactly the signal I want to probe.

In [2]:
# Count how many rows and question groups each dataset contains.
collection.frame.groupby('dataset_name').agg(
    n_rows=('dataset_name', 'size'),
    n_groups=('group_id', 'nunique'),
    n_true=('label', 'sum')
)

,n_rows,n_groups,n_true
dataset_name,,,
cities,40,10,10
larger_than,20,10,10
qa,32,8,8


In [ ]:
# Inspect one full RepE-style question group: one honest and one dishonest statement.
repeng_frame = collection.subset('repeng_truthful')
example_group_id = repeng_frame['group_id'].iloc[0]
example_group = repeng_frame.query('group_id == @example_group_id')

# The prompt column is what is sent into the language model.
example_group[['group_id', 'question', 'answer', 'label', 'prompt']]

,question,answer,label,prompt
6,Which country contains the city Paris?,Italy,False,Consider the correctness of the answer to the ...
14,Which country contains the city Paris?,Germany,False,Consider the correctness of the answer to the ...
22,Which country contains the city Paris?,France,True,Consider the correctness of the answer to the ...
23,Which country contains the city Paris?,Spain,False,Consider the correctness of the answer to the ...


## How the hidden states become features

For every prompt, the model is run with hidden states enabled. I then keep the hidden vector of the **last token**. This is a standard probing choice: by the end of the prompt, the model has already processed the question and the candidate answer, so the final position should contain a compact summary of that context.

If a model has $L$ layers and hidden size $d$, each prompt produces a tensor of shape $L \times d$. Once I choose one layer, every example becomes a vector in $\mathbb{R}^d$.

The next two notebooks use those vectors to train probes and check whether true answers score above false ones, first on one dataset and then across datasets.